# 06 --- Quality Monitoring

Quality reports, snapshots, and the append-only quality log.

## Measuring and tracking extraction quality

A single extraction result tells you what was extracted. A quality report tells you *how well* it was extracted. Over many runs, quality metrics reveal patterns that individual results cannot: is the model getting worse? Is a particular field consistently unreliable? Did a document format change break something?

### The five quality dimensions

DocFlow measures extraction quality across five dimensions, each capturing a different aspect of reliability:

1. **Completeness** — what fraction of schema fields have a non-null value? A completeness rate of 0.8 means 20% of fields came back empty. This catches omission errors.

2. **Evidence coverage** — what fraction of present fields have at least one evidence link? Fields without evidence are ungrounded claims — the model produced a value but cannot point to where it found it.

3. **Grounding rate** — of the fields with evidence, how many were verified to exist in the source text? A field with evidence that doesn't match any parsed text is a hallucination signal.

4. **Mean confidence** — the average LLM-reported confidence across fields. This is the model's self-assessment and should be treated as a soft signal, not ground truth.

5. **Auto-accept rate** — what fraction of fields passed all trust checks and can skip human review? Higher is better — it means more of your pipeline runs without human intervention.

### The quality score formula

These dimensions are combined into a single 0-1 score with weights that reflect their operational importance:

```
score = 0.15 * completeness + 0.25 * evidence + 0.25 * grounding + 0.20 * confidence + 0.15 * auto_accept
```

Evidence and grounding are weighted highest because they are the most objective measures — they don't depend on the LLM's self-assessment.

### From snapshots to drift detection

A **QualitySnapshot** is a timestamped, tagged record of a quality report. A **QualityLog** is an append-only JSONL file that accumulates snapshots over time. Together they enable drift detection: if your average quality score drops from 0.85 to 0.72 over a week, something changed — maybe the model was updated, the document format shifted, or a parser dependency broke. Tag snapshots by schema, model, date, or any other dimension to slice the data.

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

PDF_PATH = "data/sample_invoice.pdf"

from pydantic import BaseModel, Field
from typing import Optional, List


class LineItem(BaseModel):
    description: str = Field(description="Line item description")
    quantity: float = Field(description="Quantity")
    unit_price: float = Field(description="Price per unit")
    tax_rate: float = Field(description="Tax rate percentage for this item")
    amount: float = Field(description="Line item total (qty x unit_price)")


class Invoice(BaseModel):
    supplier_name: str = Field(description="Name of the supplier company")
    invoice_number: str = Field(description="Invoice reference number")
    invoice_date: str = Field(description="Date invoice was issued")
    due_date: str = Field(description="Payment due date")
    po_number: str = Field(description="Purchase order reference number")
    currency: str = Field(default="USD", description="Currency code")
    bill_to_company: str = Field(description="Billing company name")
    ship_to_company: str = Field(description="Shipping company name")
    subtotal: float = Field(description="Amount before tax")
    tax_rate: float = Field(description="Tax rate as percentage")
    tax_amount: float = Field(description="Total tax amount")
    total: float = Field(description="Grand total including tax")
    payment_terms: str = Field(description="Payment terms (e.g. Net 30)")
    line_items: List[LineItem] = Field(description="Individual line items")


from docflow import DocumentPipeline

pipeline = DocumentPipeline(parser="pymupdf", model="gemini/gemini-2.5-flash")
result = pipeline.run_sync(PDF_PATH, schema=Invoice)
print(f"Extracted {len(result.fields)} fields")

## Quality Report

`quality_report()` computes a single 0-1 score summarising completeness, grounding, evidence, confidence, and auto-accept rates.

In [ ]:
from docflow import quality_report

report = quality_report(result)

print(f"Score:             {report.score:.4f}")
print(f"Completeness:      {report.completeness_rate:.4f}")
print(f"Grounding rate:    {report.grounding_rate:.4f}")
print(f"Evidence coverage: {report.evidence_coverage:.4f}")
print(f"Mean confidence:   {report.mean_confidence:.4f}")
print(f"Auto-accept rate:  {report.auto_accept_rate:.4f}")
print(f"Correction rate:   {report.correction_rate:.4f}")
print(f"OK (>= 0.7):       {report.ok}")

## Per-field quality

Each field gets its own quality breakdown: confidence, grounding, evidence, and auto-accept status.

In [ ]:
for fname, fq in report.field_details.items():
    status = "MISSING" if fq.missing else "OK"
    print(f"  {fname:<22} [{status:7}] conf={fq.confidence:.2f} "
          f"grounded={fq.found_in_source} evidence={fq.has_evidence} "
          f"auto_accept={fq.auto_accept}")
    if fq.warning:
        print(f"  {'':22} warning: {fq.warning}")

## Warnings and worst fields

In [ ]:
print("=== Warnings ===")
if report.warnings:
    for w in report.warnings:
        print(f"  - {w}")
else:
    print("  (none)")

print()
print("=== Worst Fields ===")
if report.worst_fields:
    for f in report.worst_fields:
        print(f"  - {f}")
else:
    print("  (none -- all fields above average)")

## Score formula

The overall quality score is a weighted combination of five rates:

`score = 0.15 * completeness + 0.25 * evidence + 0.25 * grounding + 0.20 * confidence + 0.15 * auto_accept`

A score >= 0.7 is considered OK by default.

## QualitySnapshot

A snapshot is a timestamped, tagged record of a quality report -- useful for tracking quality across runs.

In [ ]:
from docflow.quality import QualitySnapshot

snapshot = QualitySnapshot.from_report(report, tags={"schema": "Invoice", "model": "gemini-flash"})

print(f"Snapshot ID:  {snapshot.snapshot_id}")
print(f"Timestamp:    {snapshot.timestamp}")
print(f"Tags:         {snapshot.tags}")
print(f"Score:        {snapshot.score:.4f}")

## QualityLog

An append-only JSONL log that persists snapshots to disk. Record snapshots and query them later by tag or recency.

In [ ]:
from docflow.quality import QualityLog
import tempfile, os

log_path = os.path.join(tempfile.mkdtemp(), "quality.jsonl")
log = QualityLog(log_path)

snap1 = log.record_sync(report, tags={"schema": "Invoice", "run": "1"})
snap2 = log.record_sync(report, tags={"schema": "Invoice", "run": "2"})
snap3 = log.record_sync(report, tags={"schema": "Receipt", "run": "3"})

print(f"Recorded 3 snapshots to {log_path}")

## Querying the log

Filter history by tags, or retrieve the most recent N snapshots.

In [ ]:
history = log.history_sync()
print(f"Total snapshots: {len(history)}")

invoice_only = log.history_sync(tags={"schema": "Invoice"})
print(f"Invoice snapshots: {len(invoice_only)}")

recent = log.history_sync(last_n=2)
print(f"Last 2 snapshots:")
for s in recent:
    print(f"  {s.snapshot_id[:8]}... score={s.score:.4f} tags={s.tags}")

## Tracking quality over time

Compare scores across runs to detect drift.

In [ ]:
history = log.history_sync(tags={"schema": "Invoice"})
scores = [s.score for s in history]

print(f"Score trend: {' -> '.join(f'{s:.4f}' for s in scores)}")

if len(scores) >= 2:
    delta = scores[-1] - scores[0]
    direction = "improving" if delta > 0 else "degrading" if delta < 0 else "stable"
    print(f"Direction: {direction} (delta: {delta:+.4f})")

## Batch quality report

`quality_report()` also accepts a list of results, returning averaged metrics and the worst fields across all results.

In [ ]:
result2 = pipeline.run_sync(PDF_PATH, schema=Invoice)

batch_report = quality_report([result, result2])

print(f"Batch score:     {batch_report.score:.4f}")
print(f"N results:       {batch_report.n_results}")
print(f"Worst fields:    {batch_report.worst_fields}")